# Lab: Mid Semester Capstone Lab – Genetic Mutation Tracking

## Case Study: Tracking Genetic Mutations Across Populations

### Scenario

You are building an AI system for a biomedical research lab studying the spread of genetic mutations across populations.

The system must:

- Navigate mutation grids (DNA regions)
- Analyze mutation pathways using graph search
- Optimize treatment strategies under adversarial mutation behavior
- Apply logical rules to detect mutation patterns
- Use first-order logic to reason about genes and organisms
- Apply probabilistic reasoning to estimate mutation risks

This system integrates multiple AI techniques into a unified pipeline.

---

## Input File Overview

This lab uses structured biological datasets:

## Input File Overview

This lab uses MULTIPLE structured datasets.

  1. genome_grid.txt  (Grid Search)

     Represents DNA region.

     Format:  Rows Cols


     S → healthy gene start  
     T → target mutation  
     \# → blocked region  
     . → normal DNA  


  2. mutation_graph.txt  (Graph Search)

     Format:

     NumberOfEdges
     geneA geneB cost direction

     direction = U (undirected) / D (directed)


  3. heuristic.txt

     gene heuristic_value


  4. coordinates.txt

     gene x y

     (Used for heuristic validation and analysis)


  5. strategy.txt  (Minimax)

     mutation_level treatment_power turn depth_limit


  6. signals.txt  (Propositional Logic)

     signal_name TRUE/FALSE


  7. rules.txt

     Logical rules in INFIX form:

     (A AND B) OR (NOT C)


  8. knowledge_base.txt  (Inference)

     Contains:

     FACT: A
     RULE: A -> B
     RULE: B -> C


  9. fol_data.txt

     Organism Gene MutationStatus


 10. probabilities.csv

     var1,var2,var3,...,probability

     (Joint distribution)
---


## Learning Objectives

- Implement BFS and DFS with path reconstruction
- Track nodes explored and compare strategies
- Implement GBFS and A* using heuristics
- Analyze optimality vs speed tradeoffs
- Model adversarial biological systems using Minimax
- Apply Alpha-Beta pruning and measure pruning impact
- Parse and evaluate infix logical expressions
- Convert logical formulas to CNF
- Perform Resolution-based theorem proving
- Implement Model Checking
- Compare inference techniques
- Implement First-Order Logic with quantifiers
- Build a probabilistic reasoning engine
- Validate probability distributions
- Integrate ALL modules into one system

## Program Requirements

## Program Requirements

You MUST implement:

SEARCH:
- bfs()
- dfs()
- gbfs()
- a_star()

GAME:
- minimax()
- alpha_beta()

LOGIC:
- infix parsing
- rule evaluation

INFERENCE:
- CNF conversion
- resolution()
- model_check()

FOL:
- universal queries (∀)
- existential queries (∃)

PROBABILITY:
- joint probability
- marginalization
- conditional probability
- Bayes rule
- independence testing

---

You MUST ALSO:

- Track nodes explored for ALL searches
- Reconstruct paths using parent dictionaries
- Compare algorithm performance
- Validate all inputs
- Handle missing data safely




## Step 1: DNA Region Navigation

Tasks:

- Implement BFS (shortest mutation path)
- Implement DFS (deep exploration)
- Maintain:
    - visited set
    - parent mapping
    - nodes explored

Output:
- Path
- Path length
- Nodes explored

In [68]:
from collections import deque

class GenomeGrid:
    def __init__(self, filename):
        self.grid = []
        self.rows = 0
        self.cols = 0
        self.start = None
        self.goal = None
        
        with open(filename, 'r') as f:
            first_line = f.readline().strip().split()
            self.rows, self.cols = int(first_line[0]), int(first_line[1])

            for r in range(self.rows):
                row = f.readline().strip().split()
                self.grid.append(row)

                for c in range(self.cols):
                    if row[c] == 'S':
                        self.start = (r, c)
                    elif row[c] == 'T':
                        self.goal = (r, c)

    def get_state(self, r, c):
        return (r, c)

    def initial_state(self):
        return self.start

    def goal_test(self, state):
        return state == self.goal

    def actions(self, state):
        r, c = state
        moves = []

        directions = [(-1, 0), (1, 0), (0, -1), (0, 1)]

        for dr, dc in directions:
            nr, nc = r + dr, c + dc

            if 0 <= nr < self.rows and 0 <= nc < self.cols:
                if self.grid[nr][nc] != '#':
                    moves.append((nr, nc))

        return moves

    def transition(self, state, action):
        return action

    def reconstruct_path(self, parent, end):
        path = []
        while end is not None:
            path.append(end)
            end = parent[end]
        path.reverse()
        return path

    def bfs(self):
        start = self.initial_state()

        queue = deque([start])
        visited = set([start])
        parent = {start: None}
        nodes_explored = 0

        while queue:
            current = queue.popleft()
            nodes_explored += 1

            if self.goal_test(current):
                path = self.reconstruct_path(parent, current)
                return path, len(path), nodes_explored

            for action in self.actions(current):
                if action not in visited:
                    visited.add(action)
                    parent[action] = current
                    queue.append(action)

        return None, 0, nodes_explored

    def dfs(self):
        start = self.initial_state()

        stack = [start]
        visited = set([start])
        parent = {start: None}
        nodes_explored = 0

        while stack:
            current = stack.pop()
            nodes_explored += 1

            if self.goal_test(current):
                path = self.reconstruct_path(parent, current)
                return path, len(path), nodes_explored

            for action in self.actions(current):
                if action not in visited:
                    visited.add(action)
                    parent[action] = current
                    stack.append(action)

        return None, 0, nodes_explored

## Step 2: Mutation Transition Graph

Tasks:

- Implement GBFS using ONLY heuristic
- Implement A* using g(n) + h(n)
- Maintain:
    - g_score
    - f_score
    - parent tracking
    - exploration order

Compare:
- cost
- nodes explored

In [69]:
import heapq

class MutationGraph:
    def __init__(self):
        self.adj = {}        
        self.heuristic = {} 
        self.coords = {}

    def add_edge(self, u, v, cost):
        if u not in self.adj:
            self.adj[u] = []
        self.adj[u].append((v, cost))

    def load_graph(self, filename):
        with open(filename, 'r') as f:
            n = int(f.readline().strip())

            for _ in range(n):
                u, v, cost, direction = f.readline().split()
                cost = int(cost)

                if direction == 'U':
                    self.add_edge(u, v, cost)
                    self.add_edge(v, u, cost)
                else:
                    self.add_edge(u, v, cost)

    def load_heuristic(self, filename):
        with open(filename, 'r') as f:
            for line in f:
                node, h = line.split()
                self.heuristic[node] = int(h)

def reconstruct_path(parent, node):
    path = []
    while node is not None:
        path.append(node)
        node = parent[node]
    path.reverse()
    return path


def compute_cost(graph, path):
    total = 0
    for i in range(len(path) - 1):
        u = path[i]
        v = path[i + 1]

        for neighbor, cost in graph.adj[u]:
            if neighbor == v:
                total += cost
                break
    return total

def gbfs(graph, start, goal):
    pq = []
    heapq.heappush(pq, (graph.heuristic[start], start))

    visited = set()
    parent = {start: None}
    nodes_explored = 0

    while pq:
        _, current = heapq.heappop(pq)

        if current in visited:
            continue

        visited.add(current)
        nodes_explored += 1

        if current == goal:
            path = reconstruct_path(parent, current)
            cost = compute_cost(graph, path)
            return path, cost, nodes_explored

        for neighbor, _ in graph.adj.get(current, []):
            if neighbor not in visited:
                parent[neighbor] = current
                heapq.heappush(pq, (graph.heuristic[neighbor], neighbor))

    return None, float('inf'), nodes_explored

def a_star(graph, start, goal):
    pq = []
    heapq.heappush(pq, (graph.heuristic[start], start))

    g_score = {start: 0}
    f_score = {start: graph.heuristic[start]}
    parent = {start: None}

    nodes_explored = 0
    visited = set()

    while pq:
        _, current = heapq.heappop(pq)

        if current in visited:
            continue

        visited.add(current)
        nodes_explored += 1

        if current == goal:
            path = reconstruct_path(parent, current)
            return path, g_score[current], nodes_explored

        for neighbor, cost in graph.adj.get(current, []):
            tentative_g = g_score[current] + cost

            if neighbor not in g_score or tentative_g < g_score[neighbor]:
                parent[neighbor] = current
                g_score[neighbor] = tentative_g
                f_score[neighbor] = tentative_g + graph.heuristic[neighbor]

                heapq.heappush(pq, (f_score[neighbor], neighbor))

    return None, float('inf'), nodes_explored

## Step 3: Mutation vs Treatment

Tasks:

- Implement Minimax recursively
- Implement Alpha-Beta pruning
- Track:
    - nodes explored
    - pruning count

Evaluation:
score = treatment_power - mutation_level

In [70]:
class GameState:
    def __init__(self, mutation, treatment, turn):
        self.mutation = mutation
        self.treatment = treatment
        self.turn = turn
    
    def is_terminal(self, depth):
        return depth == 0 or self.mutation <= 0

    def evaluate(self):
        return self.treatment - self.mutation
    
    def get_children(self):
        children = []

        if self.turn == "MAX":
            for move in [2, 4]:
                new_mutation = max(0, self.mutation - move)
                children.append(GameState(new_mutation, self.treatment, "MIN"))

        else:
            for move in [2, 4]:
                new_mutation = self.mutation + move
                children.append(GameState(new_mutation, self.treatment, "MAX"))

        return children

def minimax(state, depth, maximizing):
    nodes_explored = 0

    def recurse(state, depth, maximizing):
        nonlocal nodes_explored
        nodes_explored += 1

        if state.is_terminal(depth):
            return state.evaluate()

        if maximizing:
            max_eval = float('-inf')
            for child in state.get_children():
                eval = recurse(child, depth - 1, False)
                max_eval = max(max_eval, eval)
            return max_eval

        else:
            min_eval = float('inf')
            for child in state.get_children():
                eval = recurse(child, depth - 1, True)
                min_eval = min(min_eval, eval)
            return min_eval

    value = recurse(state, depth, maximizing)
    return value, nodes_explored

def alpha_beta(state, depth, alpha, beta, maximizing):
    nodes_explored = 0

    def recurse(state, depth, alpha, beta, maximizing):
        nonlocal nodes_explored
        nodes_explored += 1

        if state.is_terminal(depth):
            return state.evaluate()

        if maximizing:
            value = float('-inf')
            for child in state.get_children():
                value = max(value, recurse(child, depth - 1, alpha, beta, False))
                alpha = max(alpha, value)

                if beta <= alpha:
                    break

            return value

        else:
            value = float('inf')
            for child in state.get_children():
                value = min(value, recurse(child, depth - 1, alpha, beta, True))
                beta = min(beta, value)

                if beta <= alpha:
                    break

            return value

    value = recurse(state, depth, alpha, beta, maximizing)
    return value, nodes_explored

## Step 4: Mutation Rule Evaluation

Rules are provided in infix form:
Example:
    (Fever AND MutationMarker) OR NOT StableGene

Tasks:
- Parse infix expressions
- Convert to postfix
- Respect precedence (NOT > AND > OR)
- Evaluate dynamically

In [71]:
class LogicEngine:
    def __init__(self):
        self.rules = []

        self.precedence = {
            "NOT": 3,
            "AND": 2,
            "OR": 1
        }

    def is_operator(self, token):
        return token in ["NOT", "AND", "OR"]

    def parse(self, expression):
        tokens = expression.replace("(", " ( ").replace(")", " ) ").split()

        output = []
        stack = []

        for token in tokens:
            if token == '(':
                stack.append(token)

            elif token == ')':
                while stack and stack[-1] != '(':
                    output.append(stack.pop())
                stack.pop()

            elif self.is_operator(token):
                while (stack and stack[-1] != '(' and
                       self.precedence.get(stack[-1], 0) >= self.precedence[token]):
                    output.append(stack.pop())

                stack.append(token)

            else:
                output.append(token)

        while stack:
            output.append(stack.pop())

        return output

    def evaluate(self, expression, values):
        postfix = self.parse(expression)
        stack = []

        for token in postfix:
            if token == "NOT":
                val = stack.pop()
                stack.append(not val)

            elif token == "AND":
                b = stack.pop()
                a = stack.pop()
                stack.append(a and b)

            elif token == "OR":
                b = stack.pop()
                a = stack.pop()
                stack.append(a or b)

            else:
                stack.append(values[token])

        return stack[0]
    
def load_signals(filename):
    values = {}
    with open(filename, 'r') as f:
        for line in f:
            if not line.strip():
                continue
            name, val = line.split()
            values[name] = True if val == "TRUE" else False
    return values

## Step 5: Knowledge-Based Reasoning

Tasks:

- Convert rules to CNF
- Apply Resolution
- Implement Model Checking
- Compare both methods

Output:
- Whether query is provable

In [72]:
from itertools import product

def load_kb(filename):
    facts = set()
    rules = []

    with open(filename, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            if line.startswith("FACT"):
                fact = line.split(":")[1].strip()
                facts.add(fact)

            elif line.startswith("RULE"):
                rule = line.split(":")[1].strip()
                rules.append(rule)

    return facts, rules


class KnowledgeBase:
    def __init__(self, facts, rules):
        self.facts = facts
        self.rules = rules
        self.clauses = []

    def to_cnf(self, expr):
        left, right = expr.split("->")
        left = left.strip()
        right = right.strip()

        clause = []

        if "AND" in left:
            parts = [p.strip() for p in left.split("AND")]
            for p in parts:
                clause.append(f"~{p}")
        else:
            clause.append(f"~{left}")

        clause.append(right)

        return clause
    
    def build_clauses(self):
        self.clauses = []

        for fact in self.facts:
            self.clauses.append([fact])

        for rule in self.rules:
            cnf_clause = self.to_cnf(rule)
            self.clauses.append(cnf_clause)

    def negate(self, literal):
        return literal[1:] if literal.startswith("~") else "~" + literal

    def resolve(self, ci, cj):
        resolvents = []

        for li in ci:
            for lj in cj:
                if li == self.negate(lj):
                    new_clause = set(ci + cj)
                    new_clause.discard(li)
                    new_clause.discard(lj)
                    resolvents.append(list(new_clause))

        return resolvents


    def resolution(self, query):
        self.build_clauses()

        clauses = self.clauses.copy()

        clauses.append([self.negate(query)])

        new = []

        while True:
            n = len(clauses)

            for i in range(n):
                for j in range(i + 1, n):
                    resolvents = self.resolve(clauses[i], clauses[j])

                    for r in resolvents:
                        if r == []:
                            return True
                        new.append(r)

            if all(r in clauses for r in new):
                return False

            for r in new:
                if r not in clauses:
                    clauses.append(r)

    def get_symbols(self):
        symbols = set()

        for fact in self.facts:
            symbols.add(fact)

        for rule in self.rules:
            parts = rule.replace("->", " ").replace("AND", " ").split()
            for p in parts:
                symbols.add(p)

        return list(symbols)


    def evaluate_rule(self, rule, model):
        left, right = rule.split("->")
        left = left.strip()
        right = right.strip()

        if "AND" in left:
            parts = [p.strip() for p in left.split("AND")]
            lhs = all(model[p] for p in parts)
        else:
            lhs = model[left]

        return (not lhs) or model[right]


    def model_check(self, query):
        symbols = self.get_symbols()

        for values in product([True, False], repeat=len(symbols)):
            model = dict(zip(symbols, values))

            valid = True

            for fact in self.facts:
                if not model[fact]:
                    valid = False
                    break

            for rule in self.rules:
                if not self.evaluate_rule(rule, model):
                    valid = False
                    break

            if valid and not model[query]:
                return False

        return True

## Step 6: Organism–Gene Relationships

Tasks:

Implement:

∀: check condition for ALL objects  
∃: check condition for AT LEAST ONE  

Queries:

- All organisms have mutations?
- Exists organism without mutation?

In [73]:
def load_fol(filename):
    data = []

    with open(filename, 'r') as f:
        for line in f:
            if not line.strip():
                continue
            org, gene, status = line.strip().split()
            data.append({
                "organism": org,
                "gene": gene,
                "status": status
            })
    
    return data

class FOLSystem:
    def __init__(self, data):
        self.data = data

    def for_all(self, condition):
        for item in self.data:
            if not condition(item):
                return False
        return True

    def exists(self, condition):
        for item in self.data:
            if condition(item):
                return True
        return False

## Step 7: Mutation Probability Analysis

Tasks:

- Validate distributions
- Compute:
    - Joint probabilities
    - Marginals
    - Conditionals
    - Bayes Rule
- Check independence
- Apply inclusion-exclusion

In [74]:
import csv

def load_probabilities(filename):
    engine = ProbabilityEngine()

    with open(filename, 'r') as f:
        reader = csv.DictReader(f)

        for row in reader:
            prob = float(row.pop("Probability"))
            key = tuple(sorted(row.items()))
            engine.joint[key] = prob

    return engine

class ProbabilityEngine:
    def __init__(self):
        self.joint = {}

    def matches(self, entry, conditions):
        entry_dict = dict(entry)
        for k, v in conditions.items():
            if entry_dict.get(k) != v:
                return False
        return True

    def joint_prob(self, conditions):
        total = 0.0

        for entry, prob in self.joint.items():
            if self.matches(entry, conditions):
                total += prob

        return total

    def marginal(self, var):
        return self.joint_prob(var)

    def conditional(self, A, B):
        joint_AB = self.joint_prob({**A, **B})
        prob_B = self.joint_prob(B)

        if prob_B == 0:
            return 0

        return joint_AB / prob_B

    def bayes(self, A, B):
        p_B_given_A = self.conditional(B, A)
        p_A = self.joint_prob(A)
        p_B = self.joint_prob(B)

        if p_B == 0:
            return 0

        return (p_B_given_A * p_A) / p_B

    def independence(self, A, B):
        p_A = self.joint_prob(A)
        p_B = self.joint_prob(B)
        p_AB = self.joint_prob({**A, **B})

        return abs(p_AB - (p_A * p_B)) < 1e-6

## Step 8:

Run the following code cell to define the output writer function.

### Instructions

This function prints results to the console and writes them to an output file (`output.txt`)

- The output should include:
  - Search results (paths, costs, nodes explored)
  - Game strategy results
  - Logical reasoning outputs
  - FOL query results
  - Probability computations

- ❗ DO NOT modify this function
- ❗ DO NOT change the function signature
- ❗ All results from previous steps must be passed as a list of strings




In [75]:
def write_output(results, filename="output.txt"):
    output = "Genetic Mutation Analysis Results\n"
    output += "--------------------------------\n"

    for r in results:
        output += r + "\n"

    print(output)

    with open(filename, "w") as f:
        f.write(output)

## Step 8:

Implement the main function to execute the complete Genetic Mutation AI System.

### Instructions

The `main()` function integrates ALL components of this lab.

It should:

1. Load all input files
2. Initialize required classes
3. Execute:
   - Grid Search (BFS, DFS)
   - Graph Search (GBFS, A*)
   - Minimax & Alpha-Beta
   - Logical Rule Evaluation
   - Knowledge Base Inference (Resolution + Model Checking)
   - First-Order Logic Queries
   - Probability Calculations
4. Collect results from each module
5. Pass results to the output writer

In [76]:
def main():
    results = []

    # STEP 1
    grid = GenomeGrid("genome_grid.txt")

    _, bfs_len, bfs_nodes = grid.bfs()
    _, dfs_len, dfs_nodes = grid.dfs()

    results.append(f"BFS Path Length: {bfs_len}")
    results.append(f"BFS Nodes Explored: {bfs_nodes}")
    results.append(f"DFS Path Length: {dfs_len}")
    results.append(f"DFS Nodes Explored: {dfs_nodes}")

    # STEP 2
    graph = MutationGraph()
    graph.load_graph("mutation_graph.txt")
    graph.load_heuristic("heuristic.txt")

    start, goal = "G1", "G15"

    _, gbfs_cost, gbfs_nodes = gbfs(graph, start, goal)
    _, a_cost, a_nodes = a_star(graph, start, goal)

    results.append(f"GBFS Cost: {gbfs_cost}")
    results.append(f"GBFS Nodes Explored: {gbfs_nodes}")
    results.append(f"A* Cost: {a_cost}")
    results.append(f"A* Nodes Explored: {a_nodes}")

    # STEP 3
    initial_state = GameState(mutation=25, treatment=12, turn="MAX")
    depth_limit = 5

    mm_value, mm_nodes = minimax(initial_state, depth_limit, True)
    ab_value, ab_nodes = alpha_beta(
        initial_state, depth_limit, float('-inf'), float('inf'), True
    )

    results.append(f"Minimax Value: {mm_value}")
    results.append(f"Minimax Nodes: {mm_nodes}")
    results.append(f"AlphaBeta Value: {ab_value}")
    results.append(f"AlphaBeta Nodes: {ab_nodes}")

    # STEP 4
    engine = LogicEngine()
    values = load_signals("signals.txt")

    with open("rules.txt") as f:
        rules = [line.strip() for line in f if line.strip()]

    for i, rule in enumerate(rules):
        result = engine.evaluate(rule, values)
        results.append(f"Rule {i+1}: {result}")

    # STEP 5
    facts, rules = load_kb("knowledge_base.txt")
    kb = KnowledgeBase(facts, rules)

    query = "P"

    res_result = kb.resolution(query)
    mc_result = kb.model_check(query)

    results.append(f"Resolution proves {query}: {res_result}")
    results.append(f"Model Checking proves {query}: {mc_result}")

    # STEP 6
    data = load_fol("fol_data.txt")
    fol = FOLSystem(data)

    all_mut = fol.for_all(lambda x: x["status"] == "mutated")
    exists_normal = fol.exists(lambda x: x["status"] == "normal")


    results.append(f"All organisms mutated: {all_mut}")
    results.append(f"Exists organism without mutation: {exists_normal}")

    # STEP 7
    prob = load_probabilities("probabilities.csv")

    A = {"Mutation": "yes"}
    B = {"Outcome": "fail"}

    pA = prob.marginal(A)
    pB = prob.marginal(B)
    pA_given_B = prob.conditional(A, B)
    bayes_val = prob.bayes(A, B)
    indep = prob.independence(A, B)

    results.append(f"P(Mutation=yes): {pA}")
    results.append(f"P(Outcome=fail): {pB}")
    results.append(f"P(Mutation=yes | Outcome=fail): {pA_given_B}")
    results.append(f"Bayes Result: {bayes_val}")
    results.append(f"Independent: {indep}")

    write_output(results)


if __name__ == "__main__":
    main()

Genetic Mutation Analysis Results
--------------------------------
BFS Path Length: 27
BFS Nodes Explored: 171
DFS Path Length: 57
DFS Nodes Explored: 151
GBFS Cost: 38
GBFS Nodes Explored: 8
A* Cost: 18
A* Nodes Explored: 12
Minimax Value: -9
Minimax Nodes: 63
AlphaBeta Value: -9
AlphaBeta Nodes: 63
Rule 1: True
Rule 2: True
Rule 3: True
Rule 4: True
Rule 5: True
Rule 6: True
Resolution proves P: True
Model Checking proves P: True
All organisms mutated: False
Exists organism without mutation: True
P(Mutation=yes): 0.6100000000000001
P(Outcome=fail): 0.6100000000000001
P(Mutation=yes | Outcome=fail): 0.8524590163934426
Bayes Result: 0.8524590163934426
Independent: False



## Analysis Questions

1. Why might DFS be misleading in mutation navigation compared to BFS?
2. Under what conditions would GBFS outperform A* in mutation graphs?
3. How does alpha-beta pruning change decision depth in biological systems?
4. Why is CNF essential for scalable inference?
5. Compare propositional vs FOL in biological modeling.
6. How does uncertainty affect treatment planning?
7. Suggest ONE improvement using real-world biomedical data.

In [77]:
#Answers:
#
#
#
#
#
#
#